In [2]:
# ==========================================================
# CELL 1: MOUNT DRIVE + CLONE/PULL REPO
# ==========================================================
from google.colab import drive
import os, sys

drive.mount('/content/drive')

REPO_URL = "https://github.com/bestoism/skripsi-corn-label-noise"
REPO_DIR = "/content/skripsi-corn-label-noise"

if os.path.exists(REPO_DIR):
    print("🔄 Repo sudah ada, menarik update terbaru...")
    !cd {REPO_DIR} && git pull
else:
    print("⬇️  Clone repo baru...")
    !git clone {REPO_URL} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)
print(f"\n✅ Setup selesai. Working dir: {os.getcwd()}")

Mounted at /content/drive
⬇️  Clone repo baru...
Cloning into '/content/skripsi-corn-label-noise'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 66 (delta 31), reused 52 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 121.61 KiB | 707.00 KiB/s, done.
Resolving deltas: 100% (31/31), done.

✅ Setup selesai. Working dir: /content/skripsi-corn-label-noise


In [3]:
# ==========================================================
# CELL 2: INSTALL REQUIREMENTS
# ==========================================================
!pip install -q -r requirements.txt
print("✅ Dependencies terpasang.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 13.6 MB/s eta 0:00:00
✅ Dependencies terpasang.


In [4]:
# ==========================================================
# CELL 2.5: IMPORT UMUM — dipakai di banyak cell berikutnya
# ==========================================================
import os
import pandas as pd
import numpy as np

In [ ]:
# ==========================================================
# CELL 3: SCRAPING GOOGLE PLAY STORE
# ==========================================================
# ⚠️ JALANKAN SEKALI SAJA SEUMUR PROYEK. RAW_DATA_FILE TIDAK versioned
# (sumber mentah tunggal untuk semua DATA_VERSION) -- jangan dijalankan
# ulang setelah selesai.
# ==========================================================
from src import config

if os.path.exists(config.RAW_DATA_FILE):
    print(f"✅ {config.RAW_DATA_FILE} sudah ada -- scraping dilewati.")
    print("   Hapus file ini manual kalau memang mau scraping ulang dari nol.")
else:
    from scripts.scrape_google_play import main as run_scraping
    run_scraping()

In [ ]:
# ==========================================================
# CELL 3.5 (BARU): MIGRASI DATA LAMA -> DATA_VERSION="v1"
# ==========================================================
# File preprocessing lama (sebelum fix kamus slang + dedup konflik) masih
# ada di Drive dengan nama TANPA suffix versi (reviews_clean.csv,
# split_train_raw.csv, split_test.csv). Supaya bisa dipakai sebagai
# pembanding "v1" di ablasi Cell 6.6 nanti, kita salin (bukan pindah)
# ke nama baru yang sesuai skema DATA_VERSION.
#
# JALANKAN SEKALI SAJA. Kalau file lama tidak ada (proyek baru dari nol,
# belum pernah preprocessing sebelumnya), cell ini otomatis dilewati --
# artinya ablasi v1-vs-v2 di Cell 6.6 tidak relevan untukmu, langsung
# lanjut pakai v2 saja.
# ==========================================================
import shutil
from src import config

_OLD_CLEAN = os.path.join(config.DATA_PROCESSED_DIR, "reviews_clean.csv")
_OLD_TRAIN = os.path.join(config.DATA_PROCESSED_DIR, "split_train_raw.csv")
_OLD_TEST  = os.path.join(config.DATA_PROCESSED_DIR, "split_test.csv")

config.set_data_version("v1")
_migrated = 0
for old_path, new_path in [
    (_OLD_CLEAN, config.CLEAN_TEXT_FILE),
    (_OLD_TRAIN, config.TRAIN_RAW_FILE),
    (_OLD_TEST, config.TEST_FILE),
]:
    if os.path.exists(old_path) and not os.path.exists(new_path):
        shutil.copy(old_path, new_path)
        print(f"📋 Disalin: {old_path} -> {new_path}")
        _migrated += 1
    elif os.path.exists(new_path):
        print(f"✅ Sudah ada: {new_path}")
    else:
        print(f"⚠️ Tidak ditemukan (dilewati): {old_path}")

if _migrated == 0 and not os.path.exists(config.TRAIN_RAW_FILE):
    print("\nℹ️  Tidak ada data v1 untuk dimigrasikan -- proyek dimulai dari nol.")
    print("   Ablasi v1-vs-v2 (Cell 6.6) tidak relevan, lanjut langsung ke v2.")

config.set_data_version("v2")  # kembalikan ke default kerja

In [ ]:
# ==========================================================
# CELL 3.6 (BARU): MIGRASI VALIDASI MANUSIA YANG SUDAH DIISI -> v1
# ==========================================================
# Kerja manual mengisi human_verdict untuk 50 sample JANGAN sampai hilang
# cuma karena skema penamaan file berubah. Sesuaikan proxy_name di bawah
# kalau validasi manusia lama kamu itu untuk proxy selain P4.
import shutil
from src import config

_OLD_HV_SAMPLE = os.path.join(config.HUMAN_VALIDATION_DIR, "human_validation_sample.csv")
_OLD_HV_RESULT = os.path.join(config.HUMAN_VALIDATION_DIR, "human_validation_result.csv")

config.set_data_version("v1")
config.set_proxy(3)  # ganti kalau validasi manusia lama itu untuk proxy lain

for old_path, new_path in [
    (_OLD_HV_SAMPLE, config.HUMAN_VALIDATION_FILE),
    (_OLD_HV_RESULT, config.HUMAN_VALIDATION_RESULT_FILE),
]:
    if os.path.exists(old_path) and not os.path.exists(new_path):
        shutil.copy(old_path, new_path)
        print(f"📋 Disalin: {old_path} -> {new_path}")
    elif os.path.exists(new_path):
        print(f"✅ Sudah ada: {new_path}")
    else:
        print(f"⚠️ Tidak ditemukan: {old_path}")

config.set_data_version("v2")  # kembalikan ke default kerja

In [5]:
import os
from src import config
config.set_data_version("v2")
for f in [config.CLEAN_TEXT_FILE, config.TRAIN_RAW_FILE, config.TEST_FILE]:
    if os.path.exists(f):
        os.remove(f)
        print(f"🗑️ Dihapus: {f}")

📁 Data version aktif: v2
🗑️ Dihapus: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/reviews_clean__v2.csv
🗑️ Dihapus: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_train_raw__v2.csv
🗑️ Dihapus: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_test__v2.csv


In [6]:
# ==========================================================
# CELL 4: PREPROCESSING v2 -- fix kamus slang + dedup konflik teks-rating
# ==========================================================
from sklearn.model_selection import train_test_split
from src.preprocess import run_preprocessing
from src import config

config.set_data_version("v2")

if os.path.exists(config.TRAIN_RAW_FILE) and os.path.exists(config.TEST_FILE):
    print(f"✅ Split {config.DATA_VERSION} sudah ada -- preprocessing dilewati.")
    df_train = pd.read_csv(config.TRAIN_RAW_FILE)
    df_test = pd.read_csv(config.TEST_FILE)
    print(f"   Train: {len(df_train)} baris | Test: {len(df_test)} baris")
else:
    print("=" * 60)
    print(f" PREPROCESSING ({config.DATA_VERSION}) ")
    print("=" * 60)
    df_clean = run_preprocessing(config.RAW_DATA_FILE, config.CLEAN_TEXT_FILE)
    df_clean = df_clean.dropna(subset=["cleaned_text", "rating"])

    print("\n" + "=" * 60)
    print(" SPLIT DATA (80% train, 20% test, stratified by rating, seed=42) ")
    print("=" * 60)
    df_train, df_test = train_test_split(
        df_clean, test_size=0.2, random_state=42, stratify=df_clean["rating"]
    )
    df_train.to_csv(config.TRAIN_RAW_FILE, index=False)
    df_test.to_csv(config.TEST_FILE, index=False)

    print(f"✅ Train: {len(df_train)} baris -> {config.TRAIN_RAW_FILE}")
    print(f"✅ Test : {len(df_test)} baris -> {config.TEST_FILE}")

📁 Data version aktif: v2
 PREPROCESSING (v2) 
📥 Membaca data mentah dari: /content/drive/MyDrive/SKRIPSI_CORN/data/raw/all_reviews_master.csv
📏 Jumlah data awal: 10800 baris
🧹 Cleaning teks (lowercase, URL/tag, emoji, elongasi, slang)...
📊 Cakupan kamus slang: 16099/138985 kata (11.58%)
🔍 Teks identik, rating berbeda: 164 kasus
🗑️  Baris dibuang (rating minoritas dalam grup konflik): 1005
🗑️  Baris dibuang (tie, tidak ada mayoritas jelas): 181
   Total dibuang: 1186
✅ Setelah dibersihkan: 8562 baris (terbuang: 2238)
💾 Disimpan di: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/reviews_clean__v2.csv
💾 Ringkasan preprocessing -> /content/drive/MyDrive/SKRIPSI_CORN/results/preprocessing_summary__v2.csv

 SPLIT DATA (80% train, 20% test, stratified by rating, seed=42) 
✅ Train: 6849 baris -> /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_train_raw__v2.csv
✅ Test : 1713 baris -> /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_test__v2.csv


In [7]:
# ==========================================================
# CELL 4.5 (BARU): BANDINGKAN UKURAN & CAKUPAN SLANG v1 vs v2
# ==========================================================
from src import config

_summary_v1 = os.path.join(config.RESULTS_DIR, "preprocessing_summary__v1.csv")
_summary_v2 = os.path.join(config.RESULTS_DIR, "preprocessing_summary__v2.csv")

print("📊 Perbandingan preprocessing v1 (lama, ada bug) vs v2 (sudah di-fix):\n")
if os.path.exists(_summary_v1):
    display(pd.read_csv(_summary_v1))
else:
    print("   (ringkasan v1 tidak ada -- kemungkinan proyek dimulai dari nol)")

if os.path.exists(_summary_v2):
    display(pd.read_csv(_summary_v2))
else:
    print("   ⚠️ Ringkasan v2 belum ada -- pastikan Cell 4 sudah dijalankan.")

📊 Perbandingan preprocessing v1 (lama, ada bug) vs v2 (sudah di-fix):

   (ringkasan v1 tidak ada -- kemungkinan proyek dimulai dari nol)


,initial_rows,final_rows,rows_dropped,text_rating_conflicts,slang_total_words,slang_normalized_words,slang_coverage_pct
0,10800,8562,2238,164,138985,16099,11.58


In [9]:
# ==========================================================
# CELL 5: PILOT STUDY -- ABLASI PROXY 0-3 (data v2)
# ==========================================================
import pandas as pd
from src import config
from src.clean import run_confident_learning

config.set_data_version("v2")

PILOT_PROXY_IDS = [0, 1, 2, 3]  # 4 (fusion) & 6 (IndoBERTweet) dijalankan terpisah di cell lain

for pid in PILOT_PROXY_IDS:
    print(f"\n{'='*70}\n PILOT STUDY -- PROXY_ID = {pid} | DATA = {config.DATA_VERSION} \n{'='*70}")
    config.set_proxy(pid)
    try:
        run_confident_learning()
    except Exception as e:
        print(f"⚠️ Proxy {pid} gagal: {e}")
        continue

print("\n✅ Pilot study selesai.")
pilot_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
display(pilot_table[pilot_table["data_version"] == "v2"])

📁 Data version aktif: v2

 PILOT STUDY -- PROXY_ID = 0 | DATA = v2 
📌 Proxy aktif: [0] frozen_cls_lr — CLS embedding beku + LR (P1)
   Backbone: indobenchmark/indobert-base-p1 | Data: v2
 CONFIDENT LEARNING — proxy aktif: [0] frozen_cls_lr | data: v2 
📥 Memuat 6849 baris data train.

🧮 Menghitung OOF pred_probs — proxy [0] frozen_cls_lr
⚡ Memuat cache proxy [frozen_cls_lr] ...

📐 Kualitas proxy [frozen_cls_lr] (data v2):
   Exact Accuracy : 0.4303
   MAE            : 0.9366
   Off-by-1 Acc   : 0.7497
   QWK            : 0.6092

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3209 baris diflag (46.85%)
   'prune_by_noise_rate': 2735 baris diflag (39.93%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3209 / sisa 3640
   Severity-aware : buang 1351 / sisa 5498

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    1858
2     860
3     386
4     105
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIP

,proxy_id,proxy_name,proxy_desc,accuracy,mae,off_by_one,qwk,pct_flagged_noise,data_version
5,0,frozen_cls_lr,CLS embedding beku + LR (P1),0.430282,0.936633,0.749744,0.609230,46.853555,v2
6,1,frozen_meanpool_lr,Mean-pool embedding beku + LR (P2),0.430428,0.928019,0.758651,0.612447,47.233173,v2
7,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",0.445174,0.785078,0.830194,0.669671,52.577019,v2
8,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",0.451015,0.771938,0.831070,0.682986,51.700978,v2


In [5]:
# ==========================================================
# CELL 6: PROXY FINAL SEMENTARA (P4, CORN) -- data v2
# ==========================================================
# ⚠️ RESTART RUNTIME dulu sebelum cell ini kalau tadi jalankan Cell 5
# (loop pilot study), supaya config bersih.
# ==========================================================
from src import config

config.set_data_version("v2")
config.set_proxy(3)
print(f"📌 Proxy sementara: [{config.PROXY_ID}] {config.PROXY_NAME} | data {config.DATA_VERSION}")

from src.clean import run_confident_learning
df_noise, proxy_metrics = run_confident_learning()

📁 Data version aktif: v2
📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
   Backbone: indobenchmark/indobert-base-p1 | Data: v2
📁 Data version aktif: v2
📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
   Backbone: indobenchmark/indobert-base-p1 | Data: v2
📌 Proxy sementara: [3] finetuned_corn | data v2
 CONFIDENT LEARNING — proxy aktif: [3] finetuned_corn | data: v2 
📥 Memuat 6849 baris data train.

🧮 Menghitung OOF pred_probs — proxy [3] finetuned_corn
⚡ Memuat cache proxy [finetuned_corn] ...

📐 Kualitas proxy [finetuned_corn] (data v2):
   Exact Accuracy : 0.4510
   MAE            : 0.7719
   Off-by-1 Acc   : 0.8311
   QWK            : 0.6830

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3541 baris diflag (51.70%)
   'prune_by_noise_rate': 3115 baris diflag (45.48%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3541 / sisa 3308
   Severity

In [6]:
# ==========================================================
# CELL 6.5: CEK KELENGKAPAN FILE DI DRIVE
# ==========================================================
from src import config
import os

checks = {
    "Data train (v2)": config.TRAIN_RAW_FILE,
    "Data test (v2)": config.TEST_FILE,
    "Tabel ablasi proxy": config.PROXY_QUALITY_LOG_FILE,
}

all_ok = True
for label, path in checks.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"{status} {label}: {path}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ Semua file ditemukan -- aman lanjut ke Cell 6.6.")
else:
    print("\n⛔ Ada file tidak ditemukan! Cek akun Drive yang dipakai.")

✅ Data train (v2): /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_train_raw__v2.csv
✅ Data test (v2): /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_test__v2.csv
✅ Tabel ablasi proxy: /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

✅ Semua file ditemukan -- aman lanjut ke Cell 6.6.


In [7]:
# ==========================================================
# CELL 6.6 (BARU): ABLASI EFEK FIX DATA -- P4 di v1 vs v2
# ==========================================================
# Satu variabel yang berubah: DATA_VERSION. Metode proxy tetap sama (P4).
# Ini mengukur murni efek fix kamus slang + dedup konflik teks-rating,
# terpisah dari eksperimen backbone (Cell 6.7 & 6.8).
#
# CATATAN: karena cache OOF proxy 3 di v1 belum pernah dihitung dengan
# nama file baru (oof_pred_probs__finetuned_corn__v1.npy), cell ini akan
# fine-tune ulang dari nol untuk v1 (bukan dari cache) -- wajar, sekali
# saja, dan hasilnya deterministik (seed=42) sehingga sebanding dengan
# angka lama yang sudah kamu punya di laporan sebelumnya.
# ==========================================================
from src import config
from src.clean import run_confident_learning

if not os.path.exists(os.path.join(config.DATA_PROCESSED_DIR, "split_train_raw__v1.csv")):
    print("ℹ️  Data v1 tidak tersedia (lihat Cell 3.5) -- ablasi ini dilewati.")
else:
    config.set_data_version("v1")
    config.set_proxy(3)
    print(f"\n{'='*70}\n P4 DI DATA v1 (sebelum fix) \n{'='*70}")
    df_noise_v1, metrics_v1 = run_confident_learning()

    config.set_data_version("v2")
    config.set_proxy(3)
    print(f"\n{'='*70}\n P4 DI DATA v2 (sesudah fix) \n{'='*70}")
    df_noise_v2, metrics_v2 = run_confident_learning()

    print("\n" + "=" * 60)
    print(" PERBANDINGAN EFEK FIX DATA (P4, proxy sama) ")
    print("=" * 60)
    print(f"v1 (sebelum fix): Acc={metrics_v1['accuracy']:.4f} | MAE={metrics_v1['mae']:.4f} "
          f"| Off-by-1={metrics_v1['off_by_one']:.4f} | QWK={metrics_v1['qwk']:.4f}")
    print(f"v2 (sesudah fix): Acc={metrics_v2['accuracy']:.4f} | MAE={metrics_v2['mae']:.4f} "
          f"| Off-by-1={metrics_v2['off_by_one']:.4f} | QWK={metrics_v2['qwk']:.4f}")

    # kembalikan ke v2 -- default kerja untuk cell selanjutnya
    config.set_data_version("v2")
    config.set_proxy(3)

📁 Data version aktif: v1
📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
   Backbone: indobenchmark/indobert-base-p1 | Data: v1

 P4 DI DATA v1 (sebelum fix) 
 CONFIDENT LEARNING — proxy aktif: [3] finetuned_corn | data: v1 
📥 Memuat 7157 baris data train.

🧮 Menghitung OOF pred_probs — proxy [3] finetuned_corn
   [Proxy finetuned_corn] Fold 1/5 (train=5725, val=1432)...


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

...

      Epoch 1/3 - Loss: 0.5044
      Epoch 2/3 - Loss: 0.4390
      Epoch 3/3 - Loss: 0.3725
   [Proxy finetuned_corn] Fold 4/5 (train=5726, val=1431)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.5000
      Epoch 2/3 - Loss: 0.4358
      Epoch 3/3 - Loss: 0.3685
   [Proxy finetuned_corn] Fold 5/5 (train=5726, val=1431)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.5074
      Epoch 2/3 - Loss: 0.4416
      Epoch 3/3 - Loss: 0.3823

📐 Kualitas proxy [finetuned_corn] (data v1):
   Exact Accuracy : 0.4394
   MAE            : 0.8041
   Off-by-1 Acc   : 0.8223
   QWK            : 0.6692

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3649 baris diflag (50.99%)
   'prune_by_noise_rate': 3288 baris diflag (45.94%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3649 / sisa 3508
   Severity-aware : buang 1145 / sisa 6012

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2504
2     838
3     239
4      68
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

💾 Cleaned (hard)   -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_hard__finetuned_corn__v1.csv
💾 Cleaned (severe) -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_severe__finetuned_corn__v1.csv

📝 Sample validasi manus

In [8]:
# ==========================================================
# CELL 6.7: ABLASI TAMBAHAN -- P5 (FUSION SENTIMEN), data v2
# ==========================================================
import traceback
from src import config
from src.clean import run_confident_learning

config.set_data_version("v2")
config.set_proxy(4)
try:
    df_noise_p5, proxy_metrics_p5 = run_confident_learning()
except Exception as e:
    print(f"⚠️ Proxy 4 (fusion) gagal:")
    traceback.print_exc()
finally:
    config.set_proxy(3)
    print(f"\n📌 Proxy dikembalikan ke sementara: [{config.PROXY_ID}] {config.PROXY_NAME}")

📁 Data version aktif: v2
📌 Proxy aktif: [4] finetuned_corn_fusion — IndoBERT+CORN + fusi sentimen (P5)
   Backbone: indobenchmark/indobert-base-p1 | Data: v2
 CONFIDENT LEARNING — proxy aktif: [4] finetuned_corn_fusion | data: v2 
📥 Memuat 6849 baris data train.

🧮 Menghitung OOF pred_probs — proxy [4] finetuned_corn_fusion
⬇️  Memuat model sentimen: w11wo/indonesian-roberta-base-sentiment-classifier


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/808k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

...

      Epoch 1/3 - Loss: 0.5084
      Epoch 2/3 - Loss: 0.4388
      Epoch 3/3 - Loss: 0.3681
   [Proxy finetuned_corn_fusion] Fold 4/5 (train=5479, val=1370)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.4987
      Epoch 2/3 - Loss: 0.4291
      Epoch 3/3 - Loss: 0.3642
   [Proxy finetuned_corn_fusion] Fold 5/5 (train=5480, val=1369)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

      Epoch 1/3 - Loss: 0.5055
      Epoch 2/3 - Loss: 0.4366
      Epoch 3/3 - Loss: 0.3684

📐 Kualitas proxy [finetuned_corn_fusion] (data v2):
   Exact Accuracy : 0.4545
   MAE            : 0.7676
   Off-by-1 Acc   : 0.8336
   QWK            : 0.6948

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3467 baris diflag (50.62%)
   'prune_by_noise_rate': 3145 baris diflag (45.92%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3467 / sisa 3382
   Severity-aware : buang 1070 / sisa 5779

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2397
2     785
3     226
4      59
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

💾 Cleaned (hard)   -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_hard__finetuned_corn_fusion__v2.csv
💾 Cleaned (severe) -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_severe__finetuned_corn_fusion__v2.csv

📝 

In [9]:
# ==========================================================
# CELL 6.8 (BARU): ABLASI BACKBONE -- P6 (IndoBERTweet + CORN), data v2
# ==========================================================
import traceback
from src import config
from src.clean import run_confident_learning

config.set_data_version("v2")
config.set_proxy(6)
print(f"📌 Proxy: [{config.PROXY_ID}] {config.PROXY_NAME} | Backbone: {config.PRETRAINED_MODEL_NAME}")
try:
    df_noise_p6, proxy_metrics_p6 = run_confident_learning()
except Exception as e:
    print(f"⚠️ Proxy 6 (IndoBERTweet) gagal:")
    traceback.print_exc()
finally:
    config.set_proxy(3)
    print(f"\n📌 Proxy dikembalikan ke sementara: [{config.PROXY_ID}] {config.PROXY_NAME}")

📁 Data version aktif: v2
📌 Proxy aktif: [6] finetuned_corn_indobertweet — IndoBERTweet fine-tuned K-Fold, CORN loss (P6)
   Backbone: indolem/indobertweet-base-uncased | Data: v2
📌 Proxy: [6] finetuned_corn_indobertweet | Backbone: indolem/indobertweet-base-uncased
 CONFIDENT LEARNING — proxy aktif: [6] finetuned_corn_indobertweet | data: v2 
📥 Memuat 6849 baris data train.

🧮 Menghitung OOF pred_probs — proxy [6] finetuned_corn_indobertweet
   [Proxy finetuned_corn_indobertweet] Fold 1/5 (train=5479, val=1370)...


config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  445MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


      Epoch 1/3 - Loss: 0.5136
      Epoch 2/3 - Loss: 0.4399
      Epoch 3/3 - Loss: 0.3885
   [Proxy finetuned_corn_indobertweet] Fold 5/5 (train=5480, val=1369)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


      Epoch 1/3 - Loss: 0.5119
      Epoch 2/3 - Loss: 0.4427
      Epoch 3/3 - Loss: 0.3897

📐 Kualitas proxy [finetuned_corn_indobertweet] (data v2):
   Exact Accuracy : 0.4593
   MAE            : 0.7579
   Off-by-1 Acc   : 0.8368
   QWK            : 0.6956

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3265 baris diflag (47.67%)
   'prune_by_noise_rate': 2973 baris diflag (43.41%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3265 / sisa 3584
   Severity-aware : buang 958 / sisa 5891

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2307
2     719
3     185
4      54
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

💾 Cleaned (hard)   -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_hard__finetuned_corn_indobertweet__v2.csv
💾 Cleaned (severe) -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_severe__finetuned_corn_indober

In [10]:
# ==========================================================
# CELL 6.9 (BARU): RINGKASAN SEMUA PROXY DI DATA v2 -- PILIH FINAL DI SINI
# ==========================================================
from src import config

table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
table_v2 = table[table["data_version"] == "v2"].sort_values("qwk", ascending=False)
print("📊 Semua proxy yang sudah diuji di data v2, diurutkan dari QWK tertinggi:")
display(table_v2)

print("\n⚠️  TENTUKAN PROXY_ID FINAL secara manual berdasarkan tabel di atas,")
print("   lalu isi di FINAL_PROXY_ID di bawah sebelum lanjut ke Cell 7.")

📊 Semua proxy yang sudah diuji di data v2, diurutkan dari QWK tertinggi:


,proxy_id,proxy_name,proxy_desc,accuracy,mae,off_by_one,qwk,pct_flagged_noise,data_version
10,6,finetuned_corn_indobertweet,"IndoBERTweet fine-tuned K-Fold, CORN loss (P6)",0.459337,0.757921,0.836764,0.695553,47.671193,v2
9,4,finetuned_corn_fusion,IndoBERT+CORN + fusi sentimen (P5),0.454519,0.767557,0.833552,0.694756,50.620529,v2
8,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",0.451015,0.771938,0.831070,0.682986,51.700978,v2
7,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",0.445174,0.785078,0.830194,0.669671,52.577019,v2
6,1,frozen_meanpool_lr,Mean-pool embedding beku + LR (P2),0.430428,0.928019,0.758651,0.612447,47.233173,v2
5,0,frozen_cls_lr,CLS embedding beku + LR (P1),0.430282,0.936633,0.749744,0.609230,46.853555,v2



⚠️  TENTUKAN PROXY_ID FINAL secara manual berdasarkan tabel di atas,
   lalu isi di FINAL_PROXY_ID di bawah sebelum lanjut ke Cell 7.


In [6]:
# ==========================================================
# CELL 7: KUNCI PROXY FINAL + VALIDASI MANUSIA — WAJIB SEBELUM LANJUT
# ==========================================================
# GANTI angka ini sesuai proxy yang kamu pilih dari tabel Cell 6.9
FINAL_PROXY_ID = 6   # <-- EDIT DI SINI

from src import config
config.set_data_version("v2")
config.set_proxy(FINAL_PROXY_ID)
print(f"🔒 Proxy final terkunci: [{config.PROXY_ID}] {config.PROXY_NAME} | data {config.DATA_VERSION}")

print("\n⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 8 ⚠️")
print(f"1. Buka: {config.HUMAN_VALIDATION_FILE}")
print("2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:")
print("   'noise' / 'not_noise' / 'ambiguous'")
print("3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).")
print("4. Jalankan Cell 7.5 di bawah untuk cek kelengkapan + hitung agreement rate.")

📁 Data version aktif: v2
📌 Proxy aktif: [6] finetuned_corn_indobertweet — IndoBERTweet fine-tuned K-Fold, CORN loss (P6)
   Backbone: indolem/indobertweet-base-uncased | Data: v2
🔒 Proxy final terkunci: [6] finetuned_corn_indobertweet | data v2

⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 8 ⚠️
1. Buka: /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation_sample__finetuned_corn_indobertweet__v2.csv
2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:
   'noise' / 'not_noise' / 'ambiguous'
3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).
4. Jalankan Cell 7.5 di bawah untuk cek kelengkapan + hitung agreement rate.


In [12]:
# ==========================================================
# CELL 7.5: HITUNG AGREEMENT VALIDASI MANUSIA
# ==========================================================
from src.human_validation import compute_agreement

result = compute_agreement()
if result is not None:
    print("\n✅ Validasi manusia lengkap. Siap lanjut ke Cell 8 (training).")
else:
    print("\n⛔ Belum lengkap/ada error -- perbaiki dulu sebelum lanjut.")

   (file dibaca dengan delimiter ',')
 HASIL VALIDASI MANUSIA vs CLEANLAB 
Total sample direview : 50
Setuju (memang noise) : 17 (34.0%)
Ambigu                : 3 (6.0%)
Tidak setuju          : 30 (60.0%)

📊 Agreement rate per rating_diff (mendukung/menolak asumsi severity-aware pruning):
   diff=1: n=35 | noise=20.0% | not_noise=77.1% | ambiguous=2.9%
   diff=2: n=11 | noise=63.6% | not_noise=27.3% | ambiguous=9.1%
   diff=3: n=2 | noise=50.0% | not_noise=0.0% | ambiguous=50.0%
   diff=4: n=2 | noise=100.0% | not_noise=0.0% | ambiguous=0.0%
------------------------------------------------------------
Acuan pembanding (Northcutt et al., 2021, ImageNet): ~58% sample
yang direview terbukti benar-benar issue -- ini acuan wajar, bukan
standar mutlak yang harus dicapai (skala dan domain berbeda).

💾 Hasil lengkap -> /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation_result__finetuned_corn_indobertweet__v2.csv

✅ Validasi manusia lengkap. Siap lanjut ke Cell 8 (training).


In [7]:
# ==========================================================
# CELL 8: TRAINING 6 SKENARIO x 3 SEED (AUTO-RESUME)
# ==========================================================
import json
import numpy as np
from src.train import run_experiment
from src import config

# Konfirmasi eksplisit -- cegah salah backbone/data tanpa sadar
print(f"🔧 Training M1-M6 akan pakai backbone: {config.PRETRAINED_MODEL_NAME} | "
      f"data: {config.DATA_VERSION} | proxy aktif: {config.PROXY_NAME}")

scenarios = [
    {"name": "M1_Baseline_CE",        "train_path": config.TRAIN_RAW_FILE,           "loss": "ce"},
    {"name": "M2_CleanedHard_CE",     "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "ce"},
    {"name": "M3_CleanedSevere_CE",   "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "ce"},
    {"name": "M4_Baseline_CORN",      "train_path": config.TRAIN_RAW_FILE,           "loss": "corn"},
    {"name": "M5_CleanedHard_CORN",   "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "corn"},
    {"name": "M6_CleanedSevere_CORN", "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "corn"},
]

if os.path.exists(config.PROGRESS_FILE):
    with open(config.PROGRESS_FILE) as f:
        saved_progress = json.load(f)
    print("🔄 Progress sebelumnya ditemukan, melanjutkan yang belum selesai...")
else:
    saved_progress = {}
    print("🆕 Memulai training dari awal...")

all_results = []
print(f"\n🔥 {len(scenarios)} SKENARIO x {len(config.SEED_LIST)} SEED 🔥\n")

for scenario in scenarios:
    name = scenario["name"]
    metrics_list = {"mae": [], "rmse": [], "accuracy": [], "off_by_one": [], "qwk": []}
    saved_progress.setdefault(name, {})

    for seed in config.SEED_LIST:
        seed_key = str(seed)
        if seed_key in saved_progress[name]:
            print(f"⏩ {name} | seed {seed} (sudah selesai)")
            metrics = saved_progress[name][seed_key]
        else:
            metrics = run_experiment(name, scenario["train_path"], scenario["loss"], seed)
            saved_progress[name][seed_key] = metrics
            with open(config.PROGRESS_FILE, "w") as f:
                json.dump(saved_progress, f, indent=2)

        for k in metrics_list:
            metrics_list[k].append(metrics[k])

    all_results.append({
        "Model": name,
        "MAE (↓)": f"{np.mean(metrics_list['mae']):.4f} ± {np.std(metrics_list['mae']):.4f}",
        "RMSE (↓)": f"{np.mean(metrics_list['rmse']):.4f} ± {np.std(metrics_list['rmse']):.4f}",
        "Acc (↑)": f"{np.mean(metrics_list['accuracy']):.4f} ± {np.std(metrics_list['accuracy']):.4f}",
        "Off-by-1 (↑)": f"{np.mean(metrics_list['off_by_one']):.4f} ± {np.std(metrics_list['off_by_one']):.4f}",
        "QWK (↑)": f"{np.mean(metrics_list['qwk']):.4f} ± {np.std(metrics_list['qwk']):.4f}",
        "_raw_mae": np.mean(metrics_list["mae"]),
    })

df_final = pd.DataFrame(sorted(all_results, key=lambda x: x["_raw_mae"])).drop(columns=["_raw_mae"])
df_final.to_csv(config.FINAL_RESULTS_TABLE_FILE, index=False)

print("\n" + "=" * 100)
print(f" 🏆 HASIL 6 SKENARIO -- backbone: {config.PRETRAINED_MODEL_NAME} | data: {config.DATA_VERSION} 🏆")
print("=" * 100)
display(df_final)

🔧 Training M1-M6 akan pakai backbone: indolem/indobertweet-base-uncased | data: v2 | proxy aktif: finetuned_corn_indobertweet
🆕 Memulai training dari awal...

🔥 6 SKENARIO x 3 SEED 🔥


🚀 M1_Baseline_CE | Seed: 42 | Loss: CE


config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

/content/skripsi-corn-label-noise/src/train.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 7/8 - Loss: 0.1141 - Val MAE: 0.5051 | QWK: 0.8533 | Off-by-1: 0.9559 | Acc: 0.5407


/content/skripsi-corn-label-noise/src/train.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 8/8 - Loss: 0.0834 - Val MAE: 0.5085 | QWK: 0.8490 | Off-by-1: 0.9525 | Acc: 0.5424
   ⏹ Early stopping di epoch 8 (Val MAE tidak membaik 3x)
🏆 Test (dari model Val MAE terbaik=0.4847): MAE=0.7326 | RMSE=1.0873 | Acc=0.4448 | Off-by-1=0.8622 | QWK=0.7096

 🏆 HASIL 6 SKENARIO -- backbone: indolem/indobertweet-base-uncased | data: v2 🏆


,Model,MAE (↓),RMSE (↓),Acc (↑),Off-by-1 (↑),QWK (↑)
0,M4_Baseline_CORN,0.7071 ± 0.0043,1.0653 ± 0.0206,0.4629 ± 0.0110,0.8671 ± 0.0115,0.7146 ± 0.0072
1,M5_CleanedHard_CORN,0.7126 ± 0.0102,1.0907 ± 0.0111,0.4709 ± 0.0048,0.8615 ± 0.0061,0.7169 ± 0.0078
2,M2_CleanedHard_CE,0.7200 ± 0.0077,1.1002 ± 0.0034,0.4686 ± 0.0110,0.8579 ± 0.0048,0.7146 ± 0.0048
3,M6_CleanedSevere_CORN,0.7212 ± 0.0087,1.0778 ± 0.0086,0.4534 ± 0.0105,0.8640 ± 0.0042,0.7142 ± 0.0085
4,M3_CleanedSevere_CE,0.7268 ± 0.0036,1.1084 ± 0.0170,0.4701 ± 0.0105,0.8467 ± 0.0063,0.7126 ± 0.0013
5,M1_Baseline_CE,0.7431 ± 0.0145,1.1420 ± 0.0217,0.4732 ± 0.0146,0.8373 ± 0.0059,0.7030 ± 0.0109


In [8]:
# ==========================================================
# CELL 9: UJI SIGNIFIKANSI -- WILCOXON + HOLM-BONFERRONI
# ==========================================================
from src.significance import (
    collect_all_predictions,
    aggregate_errors_across_seeds,
    run_significance_test,
    run_all_effect_sizes,
)
from src import config

print(f"🔧 Menguji hasil untuk backbone: {config.PRETRAINED_MODEL_NAME} | data: {config.DATA_VERSION}")

print("📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...")
true_labels, preds_per_seed = collect_all_predictions(scenarios)

print("\n📊 Mengagregasi absolute error across seed...")
aggregated_errors = aggregate_errors_across_seeds(true_labels, preds_per_seed)

print("\n🔬 Uji Wilcoxon Signed-Rank (3 hipotesis pre-registered) + Holm-Bonferroni...")
sig_results = run_significance_test(aggregated_errors)
display(sig_results)

print("\n📐 Effect size + CI 95% (bootstrap) -- pelengkap p-value...")
effect_sizes = run_all_effect_sizes(aggregated_errors)
display(effect_sizes)

effect_sizes.to_csv(os.path.join(config.RESULTS_DIR, f"effect_sizes__{config.PROXY_NAME}__{config.DATA_VERSION}.csv"), index=False)

🔧 Menguji hasil untuk backbone: indolem/indobertweet-base-uncased | data: v2
📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...
   Memuat prediksi M1_Baseline_CE | seed 42 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   Memuat prediksi M1_Baseline_CE | seed 123 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📊 Mengagregasi absolute error across seed...

🔬 Uji Wilcoxon Signed-Rank (3 hipotesis pre-registered) + Holm-Bonferroni...

💾 Hasil uji signifikansi -> /content/drive/MyDrive/SKRIPSI_CORN/results/significance_test__finetuned_corn_indobertweet__v2.csv


,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.707141,0.743141,0.000415,0.001245,Ya
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.721152,0.707141,0.093121,0.186243,Tidak
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.712590,0.707141,0.470534,0.470534,Tidak



📐 Effect size + CI 95% (bootstrap) -- pelengkap p-value...


,model_a,model_b,mean_diff,ci_95_low,ci_95_high
0,M4_Baseline_CORN,M1_Baseline_CE,-0.035999,-0.055658,-0.016341
1,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.014011,-0.001946,0.030551
2,M5_CleanedHard_CORN,M4_Baseline_CORN,0.005449,-0.013626,0.025297


In [9]:
# ==========================================================
# CELL 10: RINGKASAN AKHIR -- SIAP DISALIN KE BAB 4
# ==========================================================
import pandas as pd
from src import config

print("=" * 70)
print(f" RINGKASAN LENGKAP -- backbone: {config.PRETRAINED_MODEL_NAME} | data: {config.DATA_VERSION} ")
print("=" * 70)

print("\n[1] TABEL ABLASI PROXY (semua data_version) -- untuk Bab 1:")
if os.path.exists(config.PROXY_QUALITY_LOG_FILE):
    proxy_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
    display(proxy_table)
else:
    print("   ⚠️ Belum ada.")

print(f"\n[2] VALIDASI MANUSIA ({config.PROXY_NAME}, {config.DATA_VERSION}):")
if os.path.exists(config.HUMAN_VALIDATION_RESULT_FILE):
    df_human = pd.read_csv(config.HUMAN_VALIDATION_RESULT_FILE)
    counts = df_human["human_verdict"].value_counts()
    total = len(df_human)
    print(f"   Total: {total} | Noise: {counts.get('noise',0)} ({counts.get('noise',0)/total*100:.1f}%) "
          f"| Not noise: {counts.get('not_noise',0)} ({counts.get('not_noise',0)/total*100:.1f}%) "
          f"| Ambiguous: {counts.get('ambiguous',0)} ({counts.get('ambiguous',0)/total*100:.1f}%)")
else:
    print("   ⚠️ Belum ada -- jalankan Cell 7.5 dulu.")

print(f"\n[3] HASIL 6 SKENARIO (Mean ± Std, 3 seed) -- untuk Bab 4:")
if os.path.exists(config.FINAL_RESULTS_TABLE_FILE):
    display(pd.read_csv(config.FINAL_RESULTS_TABLE_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 8 dulu.")

print(f"\n[4] UJI SIGNIFIKANSI -- untuk Bab 4:")
if os.path.exists(config.SIGNIFICANCE_TEST_FILE):
    display(pd.read_csv(config.SIGNIFICANCE_TEST_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 9 dulu.")

print(f"\n[5] EFFECT SIZE + CI 95% -- untuk Bab 4:")
effect_size_file = os.path.join(config.RESULTS_DIR, f"effect_sizes__{config.PROXY_NAME}__{config.DATA_VERSION}.csv")
if os.path.exists(effect_size_file):
    display(pd.read_csv(effect_size_file))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 9 dulu.")

print("\n✅ Semua file hasil tersimpan di:", config.RESULTS_DIR)

 RINGKASAN LENGKAP -- backbone: indolem/indobertweet-base-uncased | data: v2 

[1] TABEL ABLASI PROXY (semua data_version) -- untuk Bab 1:


,proxy_id,proxy_name,proxy_desc,accuracy,mae,off_by_one,qwk,pct_flagged_noise,data_version
0,0,frozen_cls_lr,CLS embedding beku + Logistic Regression (P1),0.423362,0.946207,0.748638,0.610310,48.833310,v1
1,1,frozen_meanpool_lr,Mean-pooling embedding beku + Logistic Regress...,0.422384,0.953752,0.748777,0.598690,47.953053,v1
2,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",0.438731,0.801593,0.817242,0.653276,51.893251,v1
3,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",0.439430,0.804108,0.822272,0.669240,50.985050,v1
4,4,finetuned_corn_fusion,IndoBERT+CORN + fusi sentimen eksternal (P5),0.446696,0.800475,0.815286,0.668534,51.376275,v1
5,0,frozen_cls_lr,CLS embedding beku + LR (P1),0.430282,0.936633,0.749744,0.609230,46.853555,v2
6,1,frozen_meanpool_lr,Mean-pool embedding beku + LR (P2),0.430428,0.928019,0.758651,0.612447,47.233173,v2
7,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",0.445174,0.785078,0.830194,0.669671,52.577019,v2
8,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",0.451015,0.771938,0.831070,0.682986,51.700978,v2
9,4,finetuned_corn_fusion,IndoBERT+CORN + fusi sentimen (P5),0.454519,0.767557,0.833552,0.694756,50.620529,v2



[2] VALIDASI MANUSIA (finetuned_corn_indobertweet, v2):
   Total: 50 | Noise: 17 (34.0%) | Not noise: 30 (60.0%) | Ambiguous: 3 (6.0%)

[3] HASIL 6 SKENARIO (Mean ± Std, 3 seed) -- untuk Bab 4:


,Model,MAE (↓),RMSE (↓),Acc (↑),Off-by-1 (↑),QWK (↑)
0,M4_Baseline_CORN,0.7071 ± 0.0043,1.0653 ± 0.0206,0.4629 ± 0.0110,0.8671 ± 0.0115,0.7146 ± 0.0072
1,M5_CleanedHard_CORN,0.7126 ± 0.0102,1.0907 ± 0.0111,0.4709 ± 0.0048,0.8615 ± 0.0061,0.7169 ± 0.0078
2,M2_CleanedHard_CE,0.7200 ± 0.0077,1.1002 ± 0.0034,0.4686 ± 0.0110,0.8579 ± 0.0048,0.7146 ± 0.0048
3,M6_CleanedSevere_CORN,0.7212 ± 0.0087,1.0778 ± 0.0086,0.4534 ± 0.0105,0.8640 ± 0.0042,0.7142 ± 0.0085
4,M3_CleanedSevere_CE,0.7268 ± 0.0036,1.1084 ± 0.0170,0.4701 ± 0.0105,0.8467 ± 0.0063,0.7126 ± 0.0013
5,M1_Baseline_CE,0.7431 ± 0.0145,1.1420 ± 0.0217,0.4732 ± 0.0146,0.8373 ± 0.0059,0.7030 ± 0.0109



[4] UJI SIGNIFIKANSI -- untuk Bab 4:


,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.707141,0.743141,0.000415,0.001245,Ya
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.721152,0.707141,0.093121,0.186243,Tidak
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.712590,0.707141,0.470534,0.470534,Tidak



[5] EFFECT SIZE + CI 95% -- untuk Bab 4:


,model_a,model_b,mean_diff,ci_95_low,ci_95_high
0,M4_Baseline_CORN,M1_Baseline_CE,-0.035999,-0.055658,-0.016341
1,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.014011,-0.001946,0.030551
2,M5_CleanedHard_CORN,M4_Baseline_CORN,0.005449,-0.013626,0.025297



✅ Semua file hasil tersimpan di: /content/drive/MyDrive/SKRIPSI_CORN/results
